# Dynamic Withdrawal Strategies - Daily Path Analysis

이 노트북은 Dynamic 전략의 일별 경로 데이터를 분석하고 Excel 검증을 수행합니다.

## Step 1: 라이브러리 및 데이터 로드

In [9]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


In [10]:
# 벤치마크 데이터 로드
with open('benchmark_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(f"✅ 벤치마크 데이터 로드 완료")
print(f"   데이터 크기: {data.shape}")
print(f"   날짜 범위: {data.index[0].date()} ~ {data.index[-1].date()}")

✅ 벤치마크 데이터 로드 완료
   데이터 크기: (6521, 8)
   날짜 범위: 2001-01-03 ~ 2025-12-31


## Step 2: DataPreprocessor

In [11]:
from withdrawal_backtest import DataPreprocessor, PORTFOLIOS

preprocessor = DataPreprocessor(data, add_portfolios=True)
returns_df, month_starts = preprocessor.get_data()

print(f"✅ DataPreprocessor 완료")
print(f"   Returns DataFrame: {returns_df.shape}")
print(f"   Month Starts Series: {month_starts.shape}")

=== 데이터 전처리 시작 ===
데이터 기간: 2001-01-03 ~ 2025-12-31
총 거래일: 6,521일
벤치마크: 8개

일별 수익률 계산 중... ✅ (6,520개 수익률)

수익률 통계 (연율화):
         연평균수익률   연변동성
미국성장주     12.16  19.90
국내주식      14.40  28.11
미국국채       4.18  11.06
미국외국채      3.34  11.36
신흥국달러채권    7.45  10.59
국내중기채      3.86   2.37
국내장기채      6.16   8.13
금         13.13  19.50

포트폴리오 수익률 계산:
  추가할 포트폴리오: 6개
  ✅ Port_4.0%: 연수익률 4.92%, 연변동성 2.65%
  ✅ Port_5.0%: 연수익률 6.01%, 연변동성 3.81%
  ✅ Port_6.0%: 연수익률 7.10%, 연변동성 5.39%
  ✅ Port_7.0%: 연수익률 8.28%, 연변동성 7.09%
  ✅ Port_8.0%: 연수익률 9.57%, 연변동성 8.95%
  ✅ Port_9.0%: 연수익률 10.87%, 연변동성 10.89%

월초 거래일 식별 중... ✅ (300개월)
첫 10개 월초: [datetime.date(2001, 1, 4), datetime.date(2001, 2, 1), datetime.date(2001, 3, 1), datetime.date(2001, 4, 2), datetime.date(2001, 5, 1), datetime.date(2001, 6, 1), datetime.date(2001, 7, 2), datetime.date(2001, 8, 1), datetime.date(2001, 9, 3), datetime.date(2001, 10, 1)]
✅ 전처리 완료

✅ DataPreprocessor 완료
   Returns DataFrame: (6520, 14)
   Month Starts Series: (6520,)


## Step 3: Dynamic Simulator 생성

In [12]:
from dynamic_simulator import DynamicWithdrawalSimulator

simulator = DynamicWithdrawalSimulator(returns_df, month_starts)
print(f"✅ DynamicWithdrawalSimulator 생성 완료")
print(f"   총 날짜: {len(simulator.dates)}")
print(f"   월초 개수: {np.sum(month_starts)}")

✅ DynamicWithdrawalSimulator 생성 완료
   총 날짜: 6520
   월초 개수: 300


## Step 4: 전략 파라미터 설정 (통합)

여기서 모든 전략의 파라미터를 한번에 설정합니다.

In [ ]:
# ============================================================
# 공통 파라미터
# ============================================================
start_date = '2007-10-01'  # 시작일 (금융위기 직전으로 설정 - Status 섞이도록)
test_portfolio = 'Port_5.0%'  # 포트폴리오
horizon_years = 10  # 시뮬레이션 기간 (년)
initial_wr = 0.08  # 초기 인출률 (8% - 높게 설정하여 Breach 유도)
v0 = 100.0  # 초기 NAV

# ============================================================
# Guardrails 전략 파라미터 (3가지 모드 중 택1)
# ============================================================
guardrails_params = {
    'guardrail_width': 0.20,              # ±20%
    
    # 모드 1: Cap (기본) - 둘 다 0
    # 모드 2: Return-based - return_adjustment_pct > 0
    'return_adjustment_pct': 0.0,         # 전월 수익률 기반 조정 (0 = 사용 안 함)
    'return_threshold': 0.05,             # ±5% (return-based 모드에서만 사용)
    
    # 모드 3: Guardrail-based - guardrail_adjustment_pct > 0
    'guardrail_adjustment_pct': 0.0,      # Guardrail 위반 시 조정 (0 = 사용 안 함)
    
    # Note: return_adjustment_pct와 guardrail_adjustment_pct는 동시 사용 불가 (택1)
}

# ============================================================
# Fixed Rate 전략 파라미터
# ============================================================
# Fixed는 추가 파라미터 없음

# ============================================================
# 포트폴리오 정보 출력
# ============================================================
print(f"\n{'='*60}")
print(f"전략 파라미터 설정 완료")
print(f"{'='*60}")
print(f"\n공통 설정:")
print(f"  시작일: {start_date}")
print(f"  포트폴리오: {test_portfolio}")
print(f"  시뮬레이션 기간: {horizon_years}년")
print(f"  초기 인출률: {initial_wr*100:.1f}%")
print(f"  초기 NAV: {v0:.0f}")

portfolio_config = PORTFOLIOS.get(test_portfolio)
if portfolio_config:
    print(f"\n포트폴리오 구성:")
    print(f"  목표 수익률: {portfolio_config['target_return']:.2f}%")
    print(f"  목표 변동성: {portfolio_config['target_risk']:.2f}%")
    print(f"\n  자산 구성:")
    total_weight = 0.0
    for asset_kor, weight_pct in portfolio_config['weights'].items():
        print(f"    {asset_kor:15s}: {weight_pct:6.2f}%")
        total_weight += weight_pct
    print(f"    {'-'*30}")
    print(f"    {'합계':15s}: {total_weight:6.2f}%")

# 모드 자동 판정
if guardrails_params['return_adjustment_pct'] > 0:
    mode = 'return_based'
elif guardrails_params['guardrail_adjustment_pct'] > 0:
    mode = 'guardrail_based'
else:
    mode = 'cap'

print(f"\nGuardrails 파라미터:")
print(f"  상한: {initial_wr*(1+guardrails_params['guardrail_width'])*100:.1f}%")
print(f"  하한: {initial_wr*(1-guardrails_params['guardrail_width'])*100:.1f}%")
print(f"  조정 모드: {mode}")

if mode == 'return_based':
    print(f"  [Return-based 모드]")
    print(f"  - 수익률 기반 조정: ±{guardrails_params['return_adjustment_pct']*100:.0f}%")
    print(f"  - 수익률 임계값: ±{guardrails_params['return_threshold']*100:.0f}%")
elif mode == 'guardrail_based':
    print(f"  [Guardrail-based 모드]")
    print(f"  - Guardrail 위반 시 조정: ±{guardrails_params['guardrail_adjustment_pct']*100:.0f}%")
else:
    print(f"  [Cap 모드 - 기본]")

## Step 5: Guardrails 일별 경로 조회

In [ ]:
# ============================================================
# Guardrails 전략 파라미터 출력
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrails 전략 - 인출액 계산 로직")
print(f"{'='*60}")
guardrail_width = guardrails_params['guardrail_width']
return_adjustment_pct = guardrails_params['return_adjustment_pct']
return_threshold = guardrails_params['return_threshold']
guardrail_adjustment_pct = guardrails_params['guardrail_adjustment_pct']

# 모드 자동 판정
if return_adjustment_pct > 0:
    mode = 'return_based'
elif guardrail_adjustment_pct > 0:
    mode = 'guardrail_based'
else:
    mode = 'cap'

if mode == 'cap':
    print(f"\n  📍 Cap 모드 (NAV 기준 guardrail)")
    print(f"     상한 위반 시 (current_wr > {initial_wr*(1+guardrail_width)*100:.1f}%):")
    print(f"     ➜ 새 인출액 = 현재 NAV × {initial_wr*(1+guardrail_width)*100:.1f}% / 12")
    print(f"     ➜ 즉시 상한으로 재조정 (NAV 기준)")
    print(f"\n     하한 위반 시 (current_wr < {initial_wr*(1-guardrail_width)*100:.1f}%):")
    print(f"     ➜ 새 인출액 = 현재 NAV × {initial_wr*(1-guardrail_width)*100:.1f}% / 12")
    print(f"     ➜ 즉시 하한으로 재조정 (NAV 기준)")
    print(f"\n     정상 범위 ({initial_wr*(1-guardrail_width)*100:.1f}% ≤ current_wr ≤ {initial_wr*(1+guardrail_width)*100:.1f}%):")
    print(f"     ➜ 기본 인출액 유지")
    
elif mode == 'return_based':
    print(f"\n  📍 Return-based 모드 (전월 수익률 기준)")
    print(f"     수익률 악화 시 (monthly_return < -{return_threshold*100:.0f}%):")
    print(f"     ➜ 새 인출액 = 기본액 × (1 - {return_adjustment_pct*100:.0f}%)")
    print(f"     ➜ {return_adjustment_pct*100:.0f}% 감액")
    print(f"\n     수익률 개선 시 (monthly_return > +{return_threshold*100:.0f}%):")
    print(f"     ➜ 새 인출액 = 기본액 × (1 + {return_adjustment_pct*100:.0f}%)")
    print(f"     ➜ {return_adjustment_pct*100:.0f}% 증액")
    print(f"\n     정상 범위 (-{return_threshold*100:.0f}% ≤ monthly_return ≤ +{return_threshold*100:.0f}%):")
    print(f"     ➜ 기본 인출액 유지")
    print(f"\n     ⚠️  조정 후 Guardrail 체크 적용!")
    
elif mode == 'guardrail_based':
    print(f"\n  📍 Guardrail-based 모드 (Guardrail 위반 시 조정)")
    print(f"     상한 위반 시 (current_wr > {initial_wr*(1+guardrail_width)*100:.1f}%):")
    print(f"     ➜ 새 인출액 = 기본액 × (1 - {guardrail_adjustment_pct*100:.0f}%)")
    print(f"     ➜ {guardrail_adjustment_pct*100:.0f}% 감액")
    print(f"\n     하한 위반 시 (current_wr < {initial_wr*(1-guardrail_width)*100:.1f}%):")
    print(f"     ➜ 새 인출액 = 기본액 × (1 + {guardrail_adjustment_pct*100:.0f}%)")
    print(f"     ➜ {guardrail_adjustment_pct*100:.0f}% 증액")
    print(f"\n     정상 범위:")
    print(f"     ➜ 기본 인출액 유지")
    print(f"\n     ⚠️  조정 후 Guardrail 체크 적용!")

# ============================================================
# 일별 경로 조회
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

guardrails_daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='guardrails',
    horizon_years=horizon_years,
    initial_wr=initial_wr,
    guardrail_width=guardrails_params['guardrail_width'],
    return_adjustment_pct=guardrails_params['return_adjustment_pct'],
    return_threshold=guardrails_params['return_threshold'],
    guardrail_adjustment_pct=guardrails_params['guardrail_adjustment_pct'],
    v0=v0,
    price_data=data
)

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {guardrails_daily_path_df.shape}")
print(f"  날짜 범위: {guardrails_daily_path_df['Date'].iloc[0].date()} ~ {guardrails_daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(guardrails_daily_path_df)}")

price_cols = [c for c in guardrails_daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in guardrails_daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# ============================================================
# Guardrail Status 분석
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrail Status 분포")
print(f"{'='*60}")

status_counts = guardrails_daily_path_df['Guardrail_Status'].value_counts()
print(f"\n전체 {len(guardrails_daily_path_df)}일 중:")
for status, count in status_counts.items():
    pct = count / len(guardrails_daily_path_df) * 100
    print(f"  {status:15s}: {count:4d}일 ({pct:5.2f}%)")

# 월초 데이터 필터링 (Withdrawal_Amount > 0으로 판별)
month_starts_df = guardrails_daily_path_df[guardrails_daily_path_df['Withdrawal_Amount'] > 0]
if len(month_starts_df) > 0:
    month_status_counts = month_starts_df['Guardrail_Status'].value_counts()
    print(f"\n월초 {len(month_starts_df)}개월 중:")
    for status, count in month_status_counts.items():
        pct = count / len(month_starts_df) * 100
        print(f"  {status:15s}: {count:4d}개월 ({pct:5.2f}%)")

# ============================================================
# 월초 인출액 및 Guardrail 위반 사례 확인
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrail 위반 사례")
print(f"{'='*60}")
if len(guardrails_daily_path_df[(guardrails_daily_path_df['Guardrail_Status'] != 'Normal') & (guardrails_daily_path_df['Withdrawal_Amount'] > 0)]) > 0:
    break_threshold = guardrails_daily_path_df[(guardrails_daily_path_df['Guardrail_Status'] != 'Normal') & (guardrails_daily_path_df['Withdrawal_Amount'] > 0)][
        ['Date', 'Total_NAV', 'Withdrawal_Amount', 'Current_WR', 'Guardrail_Status', 'Year_Month']
    ].head(10)
    display(len(guardrails_daily_path_df[(guardrails_daily_path_df['Guardrail_Status'] != 'Normal') & (guardrails_daily_path_df['Withdrawal_Amount'] > 0)]))
    display(break_threshold)    
else:
    print("위반 사례 없음")

# Excel 내보내기
guardrails_daily_path_df.to_excel('guardrails_path_details.xlsx', index=False)
print(f"\n✅ Excel 파일 저장 완료: guardrails_path_details.xlsx")

## Step 6: Fixed Rate 일별 경로 조회

In [ ]:
# ============================================================
# Fixed Rate 전략 파라미터 출력
# ============================================================
print(f"\n{'='*60}")
print(f"Fixed Rate 전략 - 인출액 계산 로직")
print(f"{'='*60}")
print(f"\n  📍 완전 고정 인출:")
print(f"     ➜ 첫 달: 초기 NAV × {initial_wr*100:.1f}% / 12")
print(f"     ➜ 이후: 동일 금액 영구 고정")
print(f"     ➜ Guardrail 없음, 재계산 없음")

# ============================================================
# 일별 경로 조회
# ============================================================
print(f"\n{'='*60}")
print(f"일별 경로 데이터 조회 (start_date: {start_date})")
print(f"{'='*60}")

fixed_daily_path_df = simulator.get_single_path_detail(
    portfolio=test_portfolio,
    start_date=start_date,
    strategy='fixed',
    horizon_years=horizon_years,
    initial_wr=initial_wr,
    v0=v0,
    price_data=data
)

# ============================================================
# 일별 데이터 출력
# ============================================================
print(f"\n일별 경로 DataFrame:")
print(f"  크기: {fixed_daily_path_df.shape}")
print(f"  날짜 범위: {fixed_daily_path_df['Date'].iloc[0].date()} ~ {fixed_daily_path_df['Date'].iloc[-1].date()}")
print(f"  총 거래일: {len(fixed_daily_path_df)}")

price_cols = [c for c in fixed_daily_path_df.columns if c.startswith('Price_')]
weight_cols = [c for c in fixed_daily_path_df.columns if c.startswith('Weight_')]

print(f"\n컬럼 구성:")
print(f"  Price 컬럼: {len(price_cols)}개")
print(f"  Weight 컬럼: {len(weight_cols)}개")

# ============================================================
# Guardrail Status 분석
# ============================================================
print(f"\n{'='*60}")
print(f"Guardrail Status 분포")
print(f"{'='*60}")

status_counts = fixed_daily_path_df['Guardrail_Status'].value_counts()
print(f"\n전체 {len(fixed_daily_path_df)}일 중:")
for status, count in status_counts.items():
    pct = count / len(fixed_daily_path_df) * 100
    print(f"  {status:15s}: {count:4d}일 ({pct:5.2f}%)")

# 월초 데이터 필터링 (Withdrawal_Amount > 0으로 판별)
month_starts_df = fixed_daily_path_df[fixed_daily_path_df['Withdrawal_Amount'] > 0]
if len(month_starts_df) > 0:
    month_status_counts = month_starts_df['Guardrail_Status'].value_counts()
    print(f"\n월초 {len(month_starts_df)}개월 중:")
    for status, count in month_status_counts.items():
        pct = count / len(month_starts_df) * 100
        print(f"  {status:15s}: {count:4d}개월 ({pct:5.2f}%)")

# ============================================================
# 월초 인출액 및 Guardrail 위반 사례 확인
# ============================================================
print(f"\n{'='*60}")
print(f"월초 인출액 및 Guardrail Status")
print(f"{'='*60}")
if len(fixed_daily_path_df[fixed_daily_path_df['Guardrail_Status'] != 'Normal']) > 0:
    break_threshold = fixed_daily_path_df[fixed_daily_path_df['Guardrail_Status'] != 'Normal'][
        ['Date', 'Total_NAV', 'Withdrawal_Amount', 'Current_WR', 'Guardrail_Status', 'Year_Month']
    ]
    display(break_threshold)
else:
    print("위반 사례 없음")

# Excel 내보내기
fixed_daily_path_df.to_excel('fixed_rate_path_details.xlsx', index=False)
print(f"\n✅ Excel 파일 저장 완료: fixed_rate_path_details.xlsx")

In [1]:
import pickle
import pandas as pd

with open('grid_test_results.pkl', 'rb') as f:
    results = pickle.load(f)

df = pd.DataFrame(results)
df.to_excel('grid_test_results.xlsx', index=False)
print(f"✅ {len(df)}행 저장 완료")
df.head()

✅ 15행 저장 완료


,init_wr,band,adj_on,lookback,thr_up,thr_dn,adj_up,adj_dn,x_success_rate,y_cum_withdraw_median,y_cum_withdraw_mean,p_ruin,p_terminal_fail,p_fail,cv_median,cv_mean,worst_cut_median,worst_cut_mean,p5_monthly_income,n_paths
0,0.04,0.10,False,1,0.03,-0.03,0.05,-0.05,1.0,42.152796,41.854828,0.0,0.0,0.0,0.047092,0.044806,0.023039,0.020298,0.333333,181
1,0.04,0.15,False,1,0.03,-0.03,0.05,-0.05,1.0,41.057446,40.984124,0.0,0.0,0.0,0.033861,0.029489,0.012519,0.013121,0.333333,181
2,0.04,0.20,False,1,0.03,-0.03,0.05,-0.05,1.0,40.253160,40.391405,0.0,0.0,0.0,0.012695,0.015628,0.009728,0.009081,0.333333,181
3,0.05,0.10,False,1,0.03,-0.03,0.05,-0.05,1.0,50.735991,50.882251,0.0,0.0,0.0,0.019792,0.020192,0.013145,0.014825,0.416667,181
4,0.05,0.15,False,1,0.03,-0.03,0.05,-0.05,1.0,50.036839,50.263011,0.0,0.0,0.0,0.003725,0.008618,0.008478,0.009027,0.416667,181


In [7]:
# ============================================================
# Grid Search 결과 분석 노트북용 코드
# 셀 단위로 복사하여 Jupyter에서 실행
# ============================================================
import os
os.chdir('C:\\Users\\user\\Downloads\\python\\Withdrawl\\20260211')  # pkl 파일이 있는 폴더
# %% [Cell 1] 데이터 로드
import pickle
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

with open('grid_results_full.pkl', 'rb') as f:
    results = pickle.load(f)

df = pd.DataFrame(results)
print(f"전체 결과: {len(df)}행")
print(f"\n컬럼 목록:")
for col in df.columns:
    print(f"  {col}")

# %% [Cell 2] 기본 분포 확인
print("=" * 60)
print("기본 분포")
print("=" * 60)

print(f"\nportfolio: {df['portfolio'].unique()}")
print(f"path_method: {df['path_method'].unique()}")
print(f"init_wr: {sorted(df['init_wr'].unique())}")
print(f"band: {sorted(df['band'].unique())}")
print(f"adj_on: {df['adj_on'].unique()}")

if 'strategy_type' in df.columns:
    print(f"strategy_type: {df['strategy_type'].unique()}")
    print(f"\nstrategy_type 분포:")
    print(df['strategy_type'].value_counts())

if 'fixed_baseline' in df.columns:
    print(f"\nfixed_baseline 분포:")
    print(df['fixed_baseline'].value_counts())

# %% [Cell 3] 프론티어 산점도 - 포트폴리오별
fig = px.scatter(
    df,
    x='x_success_rate',
    y='y_cum_withdraw_median',
    color='portfolio',
    hover_data=['init_wr', 'band', 'adj_on', 'p_ruin', 'cv_median'],
    title='전체 결과: 성공확률 vs 누적인출 (포트폴리오별)',
    labels={
        'x_success_rate': '성공확률',
        'y_cum_withdraw_median': '누적 인출액 (median)',
        'portfolio': '포트폴리오'
    },
    opacity=0.5,
    width=1000,
    height=600
)

# 제약선: 성공확률 90%
fig.add_vline(x=0.90, line_dash="dash", line_color="red",
              annotation_text="성공확률 90%")

fig.show()

# %% [Cell 4] Frontier 점만 표시
if 'is_frontier' in df.columns:
    frontier_df = df[df['is_frontier'] == True]
else:
    frontier_df = df  # is_frontier 컬럼 없으면 전체 사용

print(f"Frontier 점: {len(frontier_df)}개 / 전체 {len(df)}개")
print(f"\nFrontier 포트폴리오별:")
print(frontier_df['portfolio'].value_counts())

fig2 = px.scatter(
    frontier_df,
    x='x_success_rate',
    y='y_cum_withdraw_median',
    color='portfolio',
    symbol='path_method',
    hover_data=['init_wr', 'band', 'adj_on', 'p_ruin', 'cv_median'],
    title='Pareto Frontier: 성공확률 vs 누적인출',
    labels={
        'x_success_rate': '성공확률',
        'y_cum_withdraw_median': '누적 인출액 (median)',
        'portfolio': '포트폴리오'
    },
    width=1000,
    height=600
)
fig2.add_vline(x=0.90, line_dash="dash", line_color="red")
fig2.show()

# %% [Cell 5] ★ adj_on=True vs False 비교
print("=" * 60)
print("adj_on=True vs False 비교")
print("=" * 60)

adj_comparison = df.groupby('adj_on').agg({
    'x_success_rate': 'mean',
    'y_cum_withdraw_median': 'mean',
    'p_ruin': 'mean',
    'cv_median': 'mean'
}).round(4)
print(adj_comparison)

# Frontier에 adj_on=True가 얼마나 있는지
if 'is_frontier' in df.columns:
    frontier_adj = frontier_df['adj_on'].value_counts()
    print(f"\nFrontier 내 adj_on 분포:")
    print(frontier_adj)

# 동일 init_wr, band, portfolio에서 adj_on 유무에 따른 차이
# Port_5.0%, Rolling, band=0.15 기준으로 비교
mask = (df['portfolio'] == 'Port_5.0%') & (df['path_method'] == 'rolling') & (df['band'] == 0.15)
if mask.sum() > 0:
    subset = df[mask][['init_wr', 'adj_on', 'lookback', 'adj_up',
                        'x_success_rate', 'y_cum_withdraw_median', 'p_ruin', 'cv_median']]
    subset = subset.sort_values(['init_wr', 'adj_on'])
    print(f"\nPort_5.0% / Rolling / band=0.15 상세:")
    print(subset.to_string(index=False))

# %% [Cell 6] ★ 고정 인출 vs Dynamic 비교
print("=" * 60)
print("고정 인출(fixed_baseline) vs Dynamic 비교")
print("=" * 60)

# fixed_baseline 식별 (band=99 또는 strategy_type으로)
if 'strategy_type' in df.columns:
    fixed_mask = df['strategy_type'] == 'fixed_baseline'
elif 'fixed_baseline' in df.columns:
    fixed_mask = df['fixed_baseline'] == True
else:
    # band가 매우 큰 값이면 fixed_baseline으로 간주
    fixed_mask = df['band'] >= 50

dynamic_mask = ~fixed_mask

print(f"고정 인출: {fixed_mask.sum()}개")
print(f"Dynamic: {dynamic_mask.sum()}개")

if fixed_mask.sum() > 0:
    # 포트폴리오별 비교
    for port in sorted(df['portfolio'].unique()):
        print(f"\n--- {port} ---")
        
        port_fixed = df[(df['portfolio'] == port) & fixed_mask]
        port_dynamic = df[(df['portfolio'] == port) & dynamic_mask]
        
        if len(port_fixed) > 0:
            # 동일 init_wr에서 비교
            for wr in [0.05, 0.07, 0.10]:
                f_row = port_fixed[port_fixed['init_wr'] == wr]
                d_rows = port_dynamic[(port_dynamic['init_wr'] == wr) & 
                                       (port_dynamic['path_method'] == 'rolling')]
                
                if len(f_row) > 0:
                    f = f_row.iloc[0]
                    print(f"\n  init_wr={wr*100:.0f}%:")
                    print(f"    Fixed:   성공={f['x_success_rate']:.2%}, "
                          f"인출={f['y_cum_withdraw_median']:.1f}, "
                          f"ruin={f['p_ruin']:.2%}, CV={f['cv_median']:.4f}")
                    
                    if len(d_rows) > 0:
                        best_d = d_rows.loc[d_rows['y_cum_withdraw_median'].idxmax()]
                        print(f"    Dynamic: 성공={best_d['x_success_rate']:.2%}, "
                              f"인출={best_d['y_cum_withdraw_median']:.1f}, "
                              f"ruin={best_d['p_ruin']:.2%}, CV={best_d['cv_median']:.4f}, "
                              f"band={best_d['band']:.0%}")
else:
    print("⚠️  fixed_baseline 구분이 없습니다. band 값으로 확인:")
    print(df['band'].value_counts().sort_index())

# %% [Cell 7] ★ init_wr=10% 상한 문제 확인
print("=" * 60)
print("init_wr 상한 확인 (10%에서 잘리는지)")
print("=" * 60)

# 제약 통과하는 조합 중 init_wr 분포
constrained = df[(df['x_success_rate'] >= 0.90) & (df['p_ruin'] <= 0.01)]
print(f"\n제약 통과 조합: {len(constrained)}개 / {len(df)}개")

if len(constrained) > 0:
    # 포트폴리오별 최적 init_wr
    for port in sorted(constrained['portfolio'].unique()):
        port_df = constrained[constrained['portfolio'] == port]
        best = port_df.loc[port_df['y_cum_withdraw_median'].idxmax()]
        print(f"  {port}: 최적 init_wr={best['init_wr']*100:.1f}%, "
              f"band={best['band']:.0%}, 누적인출={best['y_cum_withdraw_median']:.1f}")
    
    # init_wr=10%인 비율
    at_max = constrained[constrained['init_wr'] >= 0.095]
    print(f"\n  init_wr >= 9.5%인 최적해 비율: {len(at_max)}/{len(constrained)} "
          f"({len(at_max)/len(constrained)*100:.0f}%)")
    print(f"  → 10%가 상한이면 탐색 범위 확장 필요")

# %% [Cell 8] 포트폴리오별 성공확률 vs 인출률 곡선
fig3 = px.line(
    df[(df['path_method'] == 'rolling') & (df['adj_on'] == False) & (df['band'] == 0.15)],
    x='init_wr',
    y='x_success_rate',
    color='portfolio',
    title='인출률별 성공확률 (Rolling, adj_off, band=15%)',
    labels={
        'init_wr': '초기 인출률',
        'x_success_rate': '성공확률',
        'portfolio': '포트폴리오'
    },
    width=1000,
    height=500
)
fig3.add_hline(y=0.90, line_dash="dash", line_color="red",
               annotation_text="성공확률 90%")
fig3.show()

# %% [Cell 9] CV(인출 변동성) vs 누적인출 trade-off
fig4 = px.scatter(
    df[(df['x_success_rate'] >= 0.90) & (df['p_ruin'] <= 0.01)],
    x='cv_median',
    y='y_cum_withdraw_median',
    color='portfolio',
    symbol='path_method',
    hover_data=['init_wr', 'band', 'adj_on'],
    title='제약 통과 조합: 인출 변동성 vs 누적 인출액',
    labels={
        'cv_median': '인출 변동성 (CV median)',
        'y_cum_withdraw_median': '누적 인출액 (median)',
    },
    width=1000,
    height=600
)
fig4.show()

# %% [Cell 10] 결과 요약 테이블
print("=" * 60)
print("포트폴리오별 최적 조합 요약")
print("=" * 60)

summary_rows = []
for port in sorted(df['portfolio'].unique()):
    for method in ['rolling', 'bootstrap']:
        port_df = df[(df['portfolio'] == port) & 
                     (df['path_method'] == method) &
                     (df['x_success_rate'] >= 0.90) & 
                     (df['p_ruin'] <= 0.01)]
        
        if len(port_df) > 0:
            best = port_df.loc[port_df['y_cum_withdraw_median'].idxmax()]
            summary_rows.append({
                'Portfolio': port,
                'Path': method,
                'init_wr': f"{best['init_wr']*100:.1f}%",
                'band': f"{best['band']*100:.0f}%",
                'adj_on': best['adj_on'],
                '성공확률': f"{best['x_success_rate']:.1%}",
                '누적인출': f"{best['y_cum_withdraw_median']:.1f}",
                'P_ruin': f"{best['p_ruin']:.2%}",
                'CV': f"{best['cv_median']:.4f}",
            })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

전체 결과: 3600행

컬럼 목록:
  init_wr
  band
  adj_on
  lookback
  thr_up
  thr_dn
  adj_up
  adj_dn
  x_success_rate
  y_cum_withdraw_median
  y_cum_withdraw_mean
  p_ruin
  p_terminal_fail
  p_fail
  cv_median
  cv_mean
  worst_cut_median
  worst_cut_mean
  p5_monthly_income
  n_paths
  portfolio
  path_method
  is_frontier
  is_optimal
기본 분포

portfolio: ['Port_4.0%' 'Port_5.0%' 'Port_6.0%' 'Port_7.0%' 'Port_8.0%' 'Port_9.0%']
path_method: ['rolling' 'bootstrap']
init_wr: [np.float64(0.03), np.float64(0.034999999999999996), np.float64(0.039999999999999994), np.float64(0.04499999999999999), np.float64(0.04999999999999999), np.float64(0.054999999999999986), np.float64(0.059999999999999984), np.float64(0.06499999999999997), np.float64(0.06999999999999998), np.float64(0.07499999999999998), np.float64(0.07999999999999997), np.float64(0.08499999999999996), np.float64(0.08999999999999997), np.float64(0.09499999999999997), np.float64(0.09999999999999996)]
band: [np.float64(0.05), np.float64(0.1), n

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [5]:
!pip install nbformat


   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
